# Cherimoya Chromatin Accessibility (CATv1) and ChIP-seq (BATv1)

![Cherimoya](https://proto-bio.github.io/proto-assets/images/tool/cherimoya/hero.png)

Cherimoya is a compact sequence-to-function model that predicts base-resolution DNase-seq or ATAC-seq signal from 2,114 bp of DNA ([Ramirez et al., 2026](https://doi.org/10.5281/zenodo.21535161)). CATv1 is a collection of Cherimoya models, one per ENCODE experiment: 1,518 experiments over 407 human biosamples, five cross-validation folds each. BATv1 is its ChIP-seq counterpart: 3,429 experiments of 1,139 transcription factors and other DNA-associated proteins in 183 biosamples, run here with an all-zero control track. This notebook searches the collections, scores and profiles the beta-globin locus control region HS2 enhancer in K562 and HepG2 accessibility models and in K562 CTCF and GATA1 binding models, and takes a design gradient.

In [ ]:
from proto_tools.utils.notebook_docs import (
    display_overview,
    display_api_reference,
    display_docs_section,
    display_doc_link,
    display_available_tools,
)

display_doc_link("cherimoya")
display_overview("cherimoya")
display_docs_section("cherimoya", "Background")

## Available tools

In [ ]:
display_available_tools("cherimoya")

### `run_cherimoya_catalog`

Models are named by the ENCODE experiment they were trained on. The catalog search finds experiments by biosample, assay, classification, and validation performance, and marks the default experiment of each (assay, biosample). It runs in process and needs no GPU.

In [ ]:
from proto_tools.tools.sequence_scoring.cherimoya import (
    CherimoyaCatalogConfig,
    CherimoyaCatalogInput,
    run_cherimoya_catalog,
)

In [ ]:
# Display input docs
display_api_reference("cherimoya", "input", "run_cherimoya_catalog")

# DNase-seq experiments in K562 (erythroleukemia) and HepG2 (hepatocellular carcinoma)
catalog_inputs = CherimoyaCatalogInput(biosample="^(K562|HepG2)$", assay="DNase-seq")

In [ ]:
# Display config docs
display_api_reference("cherimoya", "config", "run_cherimoya_catalog")

catalog = run_cherimoya_catalog(catalog_inputs, CherimoyaCatalogConfig(collection="catv1"))
[
    {k: getattr(e, k) for k in ("experiment_accession", "biosample", "is_default", "count_pearson_fold_0")}
    for e in catalog.experiments
]

### `run_cherimoya_score`

Given 2,114 bp DNA windows, this function predicts the total reads over the central 1,000 bp (as ln(reads + 1)) for each requested model. Below, HS2 of the beta-globin locus control region, an erythroid enhancer, is scored with the default K562 and HepG2 DNase-seq models next to a shuffled copy. Raw log counts depend on each experiment's sequencing depth, so compare a sequence with its control within a model.

In [ ]:
import random

from proto_tools.tools.sequence_scoring.cherimoya import (
    CherimoyaScoreConfig,
    CherimoyaScoreInput,
    run_cherimoya_score,
)

# hg38 chr11:5,279,665-5,281,779 (0-based, half-open): 2,114 bp centered on SCREEN cCRE EH38E1516972 (HS2)
hs2 = (
    "ACAGCGAAACCGAGTAGACTGACAAGAGGTCATTCATTCACAAATGAATACAAAAAACAGTGTCAAGTATACGTGAGGTC"
    "CTGTAATGGGTCCTGGGCCTGCAAAACCCTCACAGCTGCTAACCTCAAGAAGCTCAGTGGAGACATAACATTCATCTGAA"
    "AATTCCTTGCATGTCTGTTGAACTAATTTATCTGCCTCTCCTTAAATTTGTTCCTTTTTGTATAGTTCCTTAAGTCATGT"
    "CACATTTCTGAAACATCTTTGCCACTGTGAACTCCTACAGAAGCCAGCTTCAAAGCCATTCTTCTGGAAGCCTTCTCTAT"
    "CCCCTTGACCTCCTGTTTTTTCTCCCACAAGCATTATGTCTGTCTGTCATTGTTTTTCATCCTCTTGTAGTCCTTCACAG"
    "TTACCCACACAGGTGAACCCTTTTAGCTCTCCTGGAGGAATGTTTCTTTCCTCTCAGGATCAGAGTTGCCTACATCTTCC"
    "TAATGCACCAAGACTGGCCTGAGATGTATCCTTAAGATGAGAGCTTCCCAGTAGCACCCCAAGTCAGATCTGACCCCGTA"
    "TGTGAGCATGTGTCCTCTAACAGCACAGGCCTTTTGCCACCTAGCTGTCCAGGGGTGCCTTAAAATGGCAAACAAGGTTT"
    "GTTTTCTTTTCCTGTTTTCATGCCTTCCTCTTCCATATCCTTGTTTCATATTAATACATGTGTATAGATCCTAAAAATCT"
    "ATACACATGTATTAATAAAGCCTGATTCTGCCGCTTCTAGGTATAGAGGCCACCTGCAAGATAAATATTTGATTCACAAT"
    "AACTAATCATTCTATGGCAATTGATAACAACAAATATATATATATATATATATATACGTATATGTGTATATATATATATA"
    "TATTCAGGAAATAATATATTCTAGAATATGTCACATTCTGTCTCAGGCATCCATTTTCTTTATGATGCCGTTTGAGGTGG"
    "AGTTTTAGTCAGGTGGTCAGCTTCTCCTTTTTTTTGCCATCTGCCCTGTAAGCATCCTGCTGGGGACCCAGATAGGAGTC"
    "ATCACTCTAGGCTGAGAACATCTGGGCACACACCCTAAGCCTCAGCATGACTCATCATGACTCAGCATTGCTGTGCTTGA"
    "GCCAGAAGGTTTGCTTAGAAGGTTACACAGAACCAGAAGGCGGGGGTGGGGCACTGACCCCGACAGGGGCCTGGCCAGAA"
    "CTGCTCATGCTTGGACTATGGGAGGTCACTAATGGAGACACACAGAAATGTAACAGGAACTAAGGAAAAACTGAAGCTTA"
    "TTTAATCAGAGATGAGGATGCTGGAAGGGATAGAGGGAGCTGAGCTTGTAAAAAGTATAGTAATCATTCAGCAAATGGTT"
    "TTGAAGCACCTGCTGGATGCTAAACACTATTTTCAGTGCTTGAATCATAAATAAGAATAAAACATGTATCTTATTCCCCA"
    "CAAGAGTCCAAGTAAAAAATAACAGTTAATTATAATGTGCTCTGTCCCCCAGGCTGGAGTGCAGTGGCACGATCTCAGCT"
    "CACTGCAACCTCCGCCTCCCGGGTTCAAGCAATTCTCCTGCCTCAGCCACCCTAATAGCTGGGATTACAGGTGCACACCA"
    "CCATGCCAGGCTAATTTTTGTACTTTTTGTAGAGGCAGGGTATCACCATGTTGTCCAAGATGGTCTTGAACTCCTGAGCT"
    "CCAAGCAGTCCACCCACCTCAGCCTCCCAAAGTGCTGGGATTACAGGTGTGAGACACCATGCCCAGATTTTCCATATTTA"
    "ATAGAGGTATTTATGGGATGGGGGAAAAGAATGTTTCTCTCACTGTGGATTATTTTAGAGAGTGGAGAATGGTCAAGATT"
    "TTTTTAAAAATTAAGAAAACATAAGTTGGACCTTGAGAAATGAAAATTTATTTTTTTGTTGGAGGATACCCATTCTCTAT"
    "CTCCCATCAGGGCAAGCTGTAAGGAACTGGCTAAGACACAGTGAGACAGAGTGACTTAGTCTTAGAGGCCCCACTGGTAC"
    "CCAGATGAGAAGGCACCTTCATCACTCATCACAGTCAGCTCTGCCTTTCTCCTCTCTCCTTTCTCATCAGAAATTTCATA"
    "AGTCTACTAGGGTCAGGCAGATCACATAAGAAAA"
)
shuffled = "".join(random.Random(0).sample(hs2, len(hs2)))
len(hs2)

In [ ]:
# Display input docs
display_api_reference("cherimoya", "input", "run_cherimoya_score")

score_inputs = CherimoyaScoreInput(sequences=[hs2, shuffled])

In [ ]:
# Display config docs
display_api_reference("cherimoya", "config", "run_cherimoya_score")

score_config = CherimoyaScoreConfig(
    models=["ENCSR000EOT", "ENCSR149XIL"],  # K562 and HepG2 DNase-seq defaults
    folds=[0, 1],  # several folds are averaged
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
score = run_cherimoya_score(score_inputs, score_config)

# HS2 is predicted far above its shuffled copy in both models.
[
    {"sequence": name, **{s.model: round(s["log_counts"], 3) for s in result.scores}}
    for name, result in zip(["HS2", "shuffled HS2"], score.results)
]

### `run_cherimoya_predict`

This function returns the base-resolution profile over the 1,000 bp output window: predicted reads per base by default, or the profile shape as probabilities.

In [ ]:
import matplotlib.pyplot as plt

from proto_tools.tools.sequence_scoring.cherimoya import (
    CherimoyaPredictConfig,
    CherimoyaPredictInput,
    run_cherimoya_predict,
)

display_api_reference("cherimoya", "config", "run_cherimoya_predict")

predict = run_cherimoya_predict(
    CherimoyaPredictInput(sequences=[hs2]),
    CherimoyaPredictConfig(models=["ENCSR000EOT", "ENCSR149XIL"], device="cuda"),
)
result = predict.results[0]
fig, ax = plt.subplots(figsize=(8, 2.5))
for profile, label in zip(result.profiles, ["K562 DNase-seq", "HepG2 DNase-seq"]):
    ax.plot(range(result.output_start, result.output_end), profile.values, label=label, lw=1)
ax.set_xlabel("position in the HS2 window (bp)")
ax.set_ylabel("predicted reads per base")
ax.legend(frameon=False)
plt.show()

### BATv1: ChIP-seq models

The same tools run the BATv1 collection with `collection="batv1"`. Its models are named by ENCODE ChIP-seq experiment, so the catalog search takes a `target` filter as well as a biosample. BATv1 has no default experiments yet; `quality_flags` lists the release's reasons to use an experiment with care, and is empty for the rest. The default model (`ENCSR000EOT`) belongs to CATv1, so the models are always named for BATv1.

In [ ]:
# CTCF and GATA1 ChIP-seq experiments in K562
batv1 = run_cherimoya_catalog(
    CherimoyaCatalogInput(biosample="^K562$", target="^(CTCF|GATA1)$"),
    CherimoyaCatalogConfig(collection="batv1"),
)
[
    {k: getattr(e, k) for k in ("experiment_accession", "target", "count_pearson_fold_0", "quality_flags")}
    for e in batv1.experiments
]

In [ ]:
# BATv1 models take the experiment's control track as a second input; these tools feed an all-zero
# control, so the predictions are the sequence's own contribution. Raw log counts are on a lower
# scale than the observed track, so again compare a sequence with its control within a model.
batv1_score = run_cherimoya_score(
    score_inputs,
    CherimoyaScoreConfig(collection="batv1", models=["ENCSR000EGM", "ENCSR000EWM"], device="cuda"),
)
[
    {"sequence": name, **{s.model: round(s["log_counts"], 3) for s in result.scores}}
    for name, result in zip(["HS2", "shuffled HS2"], batv1_score.results)
]

In [ ]:
# A ChIP-seq model predicts one distribution over the plus and minus strands; the tool sums the
# two strands, so the profile has the same 1,000 bp shape as an accessibility profile.
batv1_predict = run_cherimoya_predict(
    CherimoyaPredictInput(sequences=[hs2]),
    CherimoyaPredictConfig(collection="batv1", models=["ENCSR000EGM", "ENCSR000EWM"], device="cuda"),
)
result = batv1_predict.results[0]
fig, ax = plt.subplots(figsize=(8, 2.5))
for profile, label in zip(result.profiles, ["K562 CTCF ChIP-seq", "K562 GATA1 ChIP-seq"]):
    ax.plot(range(result.output_start, result.output_end), profile.values, label=label, lw=1)
ax.set_xlabel("position in the HS2 window (bp)")
ax.set_ylabel("predicted reads per base")
ax.legend(frameon=False)
plt.show()

### `run_cherimoya_gradient`

For gradient-based design, this function takes one relaxed `(L, 4)` matrix in `A,C,G,T` order per designed candidate, places them between fixed context so the input totals 2,114 bp, and returns each requested read-count quantity with its own raw gradient. Here a 200 bp segment in the middle of HS2 is pushed toward K562 accessibility and away from HepG2 accessibility, by composing the two gradients into one objective.

In [ ]:
import numpy as np

from proto_tools.tools.sequence_scoring.cherimoya import (
    CherimoyaGradientConfig,
    CherimoyaGradientInput,
    CherimoyaLogCountsTerm,
    run_cherimoya_gradient,
)
from proto_tools.utils import SigmoidGoal, compose_terms

display_api_reference("cherimoya", "input", "run_cherimoya_gradient")

# Two random relaxed 200 bp candidates for HS2 positions 957-1157
logits = np.random.default_rng(0).normal(size=(2, 200, 4)).tolist()
gradient_inputs = CherimoyaGradientInput(inputs=logits)

In [ ]:
display_api_reference("cherimoya", "config", "run_cherimoya_gradient")

# Each term reports its raw log count and that count's own gradient, so the design goals are
# composed afterwards. Raw log counts depend on each experiment's sequencing depth, so center
# each sigmoid at its model's fold 0 prediction for unedited HS2: a design scores better as it
# moves K562 accessibility up and HepG2 accessibility down from where HS2 starts.
hs2_fold0 = {s.model: s.fold_log_counts[0] for s in score.results[0].scores}
gradient_config = CherimoyaGradientConfig(
    context_left=hs2[:957],
    context_right=hs2[1157:],
    gradient_terms=[
        CherimoyaLogCountsTerm(model="ENCSR000EOT"),
        CherimoyaLogCountsTerm(model="ENCSR149XIL"),
    ],
    device="cuda",
)
gradient = run_cherimoya_gradient(gradient_inputs, gradient_config)
objective = compose_terms(
    gradient.results[0].terms,
    goals=[
        SigmoidGoal(direction="maximize", center=hs2_fold0["ENCSR000EOT"]),
        SigmoidGoal(direction="minimize", center=hs2_fold0["ENCSR149XIL"]),
    ],
)
{
    "terms": [(t.term.model, round(t.term_value, 3)) for t in gradient.results[0].terms],
    "objective_loss": round(objective.objective_loss, 4),
    "gradient_shape": np.array(objective.gradient).shape,
}

### Export

In [ ]:
import tempfile
from pathlib import Path

export_dir = Path(tempfile.mkdtemp())
score.export("hs2_scores", export_path=export_dir, file_format="csv")
sorted(p.name for p in export_dir.iterdir())